# EXP-10: Multi-Transformer Stacking (Pure BERT Ensemble)

**Competition:** spr-2026-mammography-report-classification

**Architecture:**
- **Level 0A:** BERTimbau Base (110M) — CLS pooling
- **Level 0B:** BERTimbau Large (335M) — Mean pooling
- **Level 0C:** BioBERTpt (110M) — Attention-weighted pooling
- **Level 0D:** mDeBERTa-v3 Base (86M) — CLS pooling
- **Level 1:** LightGBM meta-learner on OOF probabilities

All models use: FGM adversarial training, weighted Focal Loss, cosine schedule, fp16.

**Setup:** Add these datasets to notebook inputs:
- `cortese/bert-base-portuguese-cased`
- `cortese/bert-large-portuguese-cased`
- `cortese/biobertpt-all`
- `cortese/mdeberta-v3-base`
- Competition data (auto-attached)

**Memory strategy:** Conservative defaults with `# SCALE_UP:` comments for tuning.

In [ ]:
import os, gc, re, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except ImportError:
    !pip install -q lightgbm
    import lightgbm as lgb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

In [ ]:
# === Config ===
def find_input(pattern):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

COMP_DIR = find_input('spr-2026-mammography')
assert COMP_DIR, f'Competition not found! {os.listdir("/kaggle/input")}'

NUM_LABELS = 7
N_SPLITS   = 5
SEED       = 42

# ======================================================================
#  MODEL CONFIGS — conservative defaults, look for SCALE_UP comments
# ======================================================================
MODELS = {
    'bertimbau_base': dict(
        path_pattern='bert-base-portuguese-cased',
        max_len=256,            # SCALE_UP: 384 or 512
        batch_size=32,
        grad_accum=1,
        epochs=3,               # SCALE_UP: 5
        lr=2e-5,
        pooling='cls',
        fgm_eps=0.5,
        enabled=True,
    ),
    'bertimbau_large': dict(
        path_pattern='bert-large-portuguese-cased',
        max_len=256,            # SCALE_UP: 384
        batch_size=8,           # Small BS to fit in 16GB VRAM
        grad_accum=4,           # Effective BS = 8*4 = 32
        epochs=3,               # SCALE_UP: 4
        lr=1e-5,                # Lower LR for large model
        pooling='mean',
        fgm_eps=0.3,            # Smaller eps for large model
        enabled=True,
    ),
    'biobertpt': dict(
        path_pattern='biobertpt',
        max_len=256,            # SCALE_UP: 384
        batch_size=32,
        grad_accum=1,
        epochs=3,               # SCALE_UP: 5
        lr=2e-5,
        pooling='attn',         # Attention-weighted pooling
        fgm_eps=0.5,
        enabled=True,
    ),
    'mdeberta': dict(
        path_pattern='mdeberta-v3-base',
        max_len=256,            # SCALE_UP: 384
        batch_size=32,
        grad_accum=1,
        epochs=3,               # SCALE_UP: 5
        lr=2e-5,
        pooling='cls',
        fgm_eps=0.5,
        enabled=True,
    ),
}

# Common training config
WARMUP_RATIO = 0.1
WD           = 0.01
FOCAL_GAMMA  = 2.0

# Resolve model paths
for name, cfg in MODELS.items():
    if cfg['enabled']:
        cfg['path'] = find_input(cfg['path_pattern'])
        if cfg['path']:
            print(f'  {name}: {cfg["path"]}')
        else:
            print(f'  {name}: NOT FOUND — disabling')
            cfg['enabled'] = False

enabled_models = {k: v for k, v in MODELS.items() if v['enabled']}
print(f'\nEnabled models: {list(enabled_models.keys())}')

In [ ]:
# === Load Data ===
train_df = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower().strip()
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text

train_df['clean'] = train_df['report'].apply(clean_text)
test_df['clean']  = test_df['report'].apply(clean_text)

groups = train_df['clean'].apply(lambda s: hashlib.md5(s.encode()).hexdigest()).values
labels = train_df['target'].values

gkf    = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(train_df, labels, groups))

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(f'Unique groups: {len(set(groups))}')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# === Core Components ===

class ReportDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class AttentionPooling(nn.Module):
    """Learned attention-weighted pooling over token hidden states."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    def forward(self, hidden_states, attention_mask):
        scores = self.attn(hidden_states).squeeze(-1)           # (B, seq_len)
        scores = scores.masked_fill(attention_mask == 0, -1e4)  # -1e4 fits in fp16 (max ~65504)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)       # (B, seq_len, 1)
        return (hidden_states * weights).sum(dim=1)             # (B, hidden)


class FlexClassifier(nn.Module):
    """Transformer classifier with flexible pooling strategy."""
    def __init__(self, model_path, num_labels, pooling='cls', drop_p=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path)
        hidden = self.bert.config.hidden_size
        self.pooling_type = pooling

        if pooling == 'attn':
            self.pool = AttentionPooling(hidden)

        self.drop = nn.Dropout(drop_p)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kw):
        kw_bert = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            # mDeBERTa does not use token_type_ids
            try:
                kw_bert['token_type_ids'] = token_type_ids
                out = self.bert(**kw_bert)
            except TypeError:
                del kw_bert['token_type_ids']
                out = self.bert(**kw_bert)
        else:
            out = self.bert(**kw_bert)

        hs = out.last_hidden_state  # (B, seq_len, hidden)

        if self.pooling_type == 'cls':
            pooled = hs[:, 0]
        elif self.pooling_type == 'mean':
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hs * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        elif self.pooling_type == 'attn':
            pooled = self.pool(hs, attention_mask)
        else:
            pooled = hs[:, 0]

        return self.classifier(self.drop(pooled))


class FGM:
    def __init__(self, model, eps=0.5, emb_name='word_embeddings'):
        self.model = model
        self.eps = eps
        self.emb_name = emb_name
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    param.data.add_(self.eps * param.grad / norm)
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha[labels] * ((1 - pt) ** self.gamma) * ce).mean()


def compute_class_weights(y, nc=7):
    c = np.bincount(y, minlength=nc).astype(float)
    w = len(y) / (nc * c + 1e-6)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def vram_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e6
    return 0

print('Components defined.')

In [ ]:
# === Generic Training Function ===

def train_model_cv(model_name, cfg):
    """Train a transformer model with GroupKFold CV.
    Returns OOF probabilities (n_train, 7) and test probabilities (n_test, 7)."""
    print(f'\n{"#"*60}')
    print(f'# Training: {model_name} (pooling={cfg["pooling"]}, bs={cfg["batch_size"]}x{cfg["grad_accum"]}, ep={cfg["epochs"]})')
    print(f'{"#"*60}')
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(cfg['path'])
    oof_probs  = np.zeros((len(train_df), NUM_LABELS))
    test_probs = np.zeros((len(test_df), NUM_LABELS))
    fold_scores = []

    # Tokenize test once
    test_enc = tokenizer(
        test_df['report'].tolist(), truncation=True,
        padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
    )

    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f'\n--- {model_name} Fold {fold} (VRAM: {vram_mb():.0f} MB) ---')

        tr_enc = tokenizer(
            train_df.iloc[tr_idx]['report'].tolist(), truncation=True,
            padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
        )
        va_enc = tokenizer(
            train_df.iloc[va_idx]['report'].tolist(), truncation=True,
            padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
        )

        tr_ds = ReportDataset(tr_enc, labels[tr_idx])
        va_ds = ReportDataset(va_enc)
        te_ds = ReportDataset(test_enc)

        tr_loader = DataLoader(tr_ds, batch_size=cfg['batch_size'], shuffle=True,
                               num_workers=2, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=64, shuffle=False,
                               num_workers=2, pin_memory=True)
        te_loader = DataLoader(te_ds, batch_size=64, shuffle=False,
                               num_workers=2, pin_memory=True)

        model = FlexClassifier(cfg['path'], NUM_LABELS, cfg['pooling']).to(DEVICE)
        # Force all params to fp32 — mDeBERTa-v3 stores embeddings in fp16 which
        # causes "Attempting to unscale FP16 gradients" with GradScaler.
        model = model.float()
        print(f'  Model loaded. VRAM: {vram_mb():.0f} MB')

        alpha = compute_class_weights(labels[tr_idx]).to(DEVICE)
        criterion = WeightedFocalLoss(alpha, gamma=FOCAL_GAMMA)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=WD)
        total_steps = len(tr_loader) * cfg['epochs'] // cfg['grad_accum']
        warmup_steps = int(total_steps * WARMUP_RATIO)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        fgm = FGM(model, eps=cfg['fgm_eps'])
        scaler = torch.cuda.amp.GradScaler()

        best_f1 = 0
        best_oof = None
        best_state = None

        for epoch in range(cfg['epochs']):
            model.train()
            total_loss = 0
            optimizer.zero_grad()

            for step, batch in enumerate(tr_loader):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                ttype_ids = batch.get('token_type_ids')
                if ttype_ids is not None:
                    ttype_ids = ttype_ids.to(DEVICE)
                lbl = batch['labels'].to(DEVICE)

                # Forward
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attn_mask, ttype_ids)
                    loss = criterion(logits, lbl) / cfg['grad_accum']
                scaler.scale(loss).backward()

                # FGM adversarial
                fgm.attack()
                with torch.cuda.amp.autocast():
                    logits_adv = model(input_ids, attn_mask, ttype_ids)
                    loss_adv = criterion(logits_adv, lbl) / cfg['grad_accum']
                scaler.scale(loss_adv).backward()
                fgm.restore()

                total_loss += loss.item() * cfg['grad_accum']

                # Step optimizer every grad_accum steps
                if (step + 1) % cfg['grad_accum'] == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

            # Validation
            model.eval()
            va_probs = []
            with torch.no_grad():
                for batch in va_loader:
                    input_ids = batch['input_ids'].to(DEVICE)
                    attn_mask = batch['attention_mask'].to(DEVICE)
                    ttype_ids = batch.get('token_type_ids')
                    if ttype_ids is not None:
                        ttype_ids = ttype_ids.to(DEVICE)
                    with torch.cuda.amp.autocast():
                        logits = model(input_ids, attn_mask, ttype_ids)
                    va_probs.append(F.softmax(logits, dim=-1).cpu().float().numpy())

            va_probs = np.vstack(va_probs)
            f1 = f1_score(labels[va_idx], va_probs.argmax(1), average='macro')
            avg_loss = total_loss / len(tr_loader)
            print(f'  Ep {epoch} | loss={avg_loss:.4f} | val_f1={f1:.4f} | VRAM={vram_mb():.0f}MB')

            if f1 > best_f1:
                best_f1 = f1
                best_oof = va_probs.copy()
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Store OOF
        oof_probs[va_idx] = best_oof
        fold_scores.append(best_f1)

        # Test predictions with best checkpoint
        model.load_state_dict(best_state)
        model.eval()
        te_probs = []
        with torch.no_grad():
            for batch in te_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                ttype_ids = batch.get('token_type_ids')
                if ttype_ids is not None:
                    ttype_ids = ttype_ids.to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attn_mask, ttype_ids)
                te_probs.append(F.softmax(logits, dim=-1).cpu().float().numpy())
        test_probs += np.vstack(te_probs) / N_SPLITS

        # Cleanup
        del model, optimizer, scheduler, scaler, fgm, criterion, best_state
        del tr_enc, va_enc, tr_ds, va_ds, tr_loader, va_loader
        gc.collect()
        torch.cuda.empty_cache()

    oof_f1 = f1_score(labels, oof_probs.argmax(1), average='macro')
    elapsed = (time.time() - t0) / 60
    print(f'\n{model_name} OOF F1: {oof_f1:.4f} | Mean fold: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} | Time: {elapsed:.1f} min')

    return oof_probs, test_probs, oof_f1

In [ ]:
# === Train All Models ===
all_oof   = {}   # model_name -> (n_train, 7)
all_test  = {}   # model_name -> (n_test, 7)
all_f1    = {}   # model_name -> float

for name, cfg in enabled_models.items():
    oof, test, f1 = train_model_cv(name, cfg)
    all_oof[name]  = oof
    all_test[name] = test
    all_f1[name]   = f1

print('\n' + '='*60)
print('Individual Model Results:')
for name, f1 in all_f1.items():
    print(f'  {name:25s} OOF F1 = {f1:.4f}')
print('='*60)

---
## Level 1 — Meta-Learner: LightGBM Stacking

In [ ]:
# === Build Stacking Features ===
if not all_oof:
    raise RuntimeError(
        "Nenhum modelo foi treinado. Verifique se os datasets estão attachados ao notebook:\n"
        "  - cortese/bert-base-portuguese-cased\n"
        "  - dennisfong/neuralmindbert-large-portuguese-cased\n"
        "  - eduvenson/pucpr-biobertpt-all\n"
        "  - cortese/mdeberta-v3-base"
    )

model_names = list(all_oof.keys())
n_models    = len(model_names)

X_stack_train = np.hstack([all_oof[m] for m in model_names])   # (n_train, 7*n_models)
X_stack_test  = np.hstack([all_test[m] for m in model_names])  # (n_test,  7*n_models)

print(f'Stacking features: {X_stack_train.shape[1]} ({n_models} models x {NUM_LABELS} classes)')
print(f'Models: {model_names}')

In [ ]:
# === Method 1: Weighted Average (grid search) ===
if n_models == 4:
    best_w_f1 = 0
    best_weights = None
    for w0 in np.arange(0.15, 0.55, 0.05):
        for w1 in np.arange(0.15, 0.55, 0.05):
            for w2 in np.arange(0.05, 0.40, 0.05):
                w3 = 1.0 - w0 - w1 - w2
                if w3 < 0.05:
                    continue
                ens = sum(w * all_oof[m] for w, m in zip([w0,w1,w2,w3], model_names))
                f1 = f1_score(labels, ens.argmax(1), average='macro')
                if f1 > best_w_f1:
                    best_w_f1 = f1
                    best_weights = [w0, w1, w2, w3]
    print(f'Weighted avg OOF F1: {best_w_f1:.4f}')
    for m, w in zip(model_names, best_weights):
        print(f'  {m}: {w:.2f}')
else:
    # Fallback: equal weights
    ens = sum(all_oof[m] for m in model_names) / n_models
    best_w_f1 = f1_score(labels, ens.argmax(1), average='macro')
    best_weights = [1.0/n_models] * n_models
    print(f'Equal avg OOF F1: {best_w_f1:.4f}')

In [ ]:
# === Method 2: LightGBM Meta-Learner ===
META_PARAMS = dict(
    objective='multiclass', num_class=NUM_LABELS, metric='multi_logloss',
    learning_rate=0.05, num_leaves=15, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=3,
    class_weight='balanced', verbose=-1, seed=SEED,
)

meta_oof  = np.zeros((len(train_df), NUM_LABELS))
meta_test = np.zeros((len(test_df), NUM_LABELS))

for fold, (tr_idx, va_idx) in enumerate(splits):
    dtrain = lgb.Dataset(X_stack_train[tr_idx], label=labels[tr_idx])
    dvalid = lgb.Dataset(X_stack_train[va_idx], label=labels[va_idx], reference=dtrain)

    model = lgb.train(
        META_PARAMS, dtrain, num_boost_round=500,
        valid_sets=[dvalid], callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(100),
        ],
    )
    meta_oof[va_idx] = model.predict(X_stack_train[va_idx])
    meta_test += model.predict(X_stack_test) / N_SPLITS

meta_f1 = f1_score(labels, meta_oof.argmax(1), average='macro')
print(f'\nLGB Meta-Learner OOF F1: {meta_f1:.4f}')

In [ ]:
# === Method 3: Ridge Meta-Learner ===
from sklearn.linear_model import RidgeClassifierCV

ridge_oof_preds = np.zeros(len(train_df), dtype=int)
ridge_test_dfs  = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    ridge = RidgeClassifierCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], class_weight='balanced')
    ridge.fit(X_stack_train[tr_idx], labels[tr_idx])
    ridge_oof_preds[va_idx] = ridge.predict(X_stack_train[va_idx])
    ridge_test_dfs.append(ridge.decision_function(X_stack_test))

ridge_f1 = f1_score(labels, ridge_oof_preds, average='macro')
ridge_test_avg = np.mean(ridge_test_dfs, axis=0)
print(f'Ridge Meta-Learner OOF F1: {ridge_f1:.4f}')

In [ ]:
# === Pick Best Ensemble Strategy ===
print('\n' + '='*60)
print('FINAL COMPARISON')
print('='*60)
for name, f1 in all_f1.items():
    print(f'  {name:25s} {f1:.4f}')
print(f'  {"Weighted Average":25s} {best_w_f1:.4f}')
print(f'  {"LGB Meta-Learner":25s} {meta_f1:.4f}')
print(f'  {"Ridge Meta-Learner":25s} {ridge_f1:.4f}')

results = {
    'weighted_avg': (best_w_f1, 'wavg'),
    'lgb_meta':     (meta_f1,   'lgb'),
    'ridge_meta':   (ridge_f1,  'ridge'),
}
best_name, (best_f1, best_tag) = max(results.items(), key=lambda x: x[1][0])
print(f'\n>>> Best: {best_name} (F1={best_f1:.4f})')

if best_tag == 'wavg':
    final_probs = sum(w * all_test[m] for w, m in zip(best_weights, model_names))
    final_preds = final_probs.argmax(axis=1)
elif best_tag == 'lgb':
    final_preds = meta_test.argmax(axis=1)
else:
    final_preds = ridge_test_avg.argmax(axis=1)

In [ ]:
# === Generate Submission ===
sub = pd.DataFrame({'ID': test_df['ID'], 'target': final_preds})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub['target'].value_counts().sort_index())
print(f'\nSubmission saved: {sub.shape}')
sub.head()